In [1]:
from PIL import Image
import pytesseract

In [2]:
pytesseract.pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

In [3]:
print(pytesseract.get_languages(config=''))

['eng', 'osd', 'snum']


In [ ]:
image = Image.open("/Users/mochi/Downloads/ilovepdf_pages-to-jpg/CamScanner 4-29-26 21.13_pages-to-jpg-0001.jpg")

In [20]:
extracted_text = pytesseract.image_to_string(image, lang='eng')

In [21]:
extracted_text

''

In [25]:
config = r'--psm 6 --oem 3 -l eng preserve_interword_spaces=1'
text = pytesseract.image_to_string(image, config=config)

In [26]:
text

'¥ ~ P Municipality of Infanta\nOffice of the Municipal Mayor\nBACON Pr IPMAS = ©\nBusiness Permit\nTo whom it may concern,\nPursuant to the revenue code of this Municipality/City, after payment of taxes, fees and charges, etc., and compliance\nwith existing requirements, Permit is hereby granted to the herein Taxpayer. | |\nBUSINESS NAME | OWNER\'SNAME —-BUSINESSIDNO, BUSINESS PLATE BUSINESS PERMIT NO.\n— ae —— hoe ——— oon i 1 — — ——-——+ - * ee ar be ae ee eee a — — .\nLMAS ENTERPRISES SERAORA ——-P045620-00179 OG GT ; 2026-0405620000-0306\n7 , one ~ - . i . ; - .\nDATE ISSUED _TYPEOF BUSINESS © PAYMENTMODE OFFICIAL RECEIPT — OR DATE\n———— i ininenlinmandmestednanmeentiine nadia | a wee ee oe a -— erates | — +++ = aoe 37 ~ Ry eng pee\n2026-02-06 | Sole Proprietorshi hewaaal : 9783465 Jan. 18, 2026\nie 9783467 Jan. 18, 2026\nBUSINESS ADDRESS | peeteean | LINE OF BUSINESS\n| Se | a eee\nPUROK LANGKA GUMIAN, INFANTA, QUEZON Renewal | OTHER WHOLESALE OF FOOD, BEVERAGES AND TOBACCO\nVALID U

In [ ]:
import cv2
import numpy as np

def clean_for_tesseract(image_path):
    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # 1. Increase contrast and binarize (Adaptive Thresholding)
    # This turns shadows/stamps into white and keeps text black
    clean_img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                      cv2.THRESH_BINARY, 11, 2)
    
    # 2. Denoising (Removes tiny dots/pepper noise)
    kernel = np.ones((1, 1), np.uint8)
    clean_img = cv2.dilate(clean_img, kernel, iterations=1)
    clean_img = cv2.erode(clean_img, kernel, iterations=1)
    
    return clean_img

# Test it
cleaned = clean_for_tesseract(image_path="/Users/mochi/Downloads/ilovepdf_pages-to-jpg/CamScanner 4-29-26 21.13_pages-to-jpg-0001.jpg")
cv2.imwrite("/Users/mochi/Downloads/ilovepdf_pages-to-jpg/business_permit_test_cleaned.jpg", cleaned)

True

In [ ]:
import pytesseract
from pytesseract import Output

d = pytesseract.image_to_data(image, output_type=Output.DICT)
n_boxes = len(d['level'])
lines = {}

for i in range(n_boxes):
    if int(d['conf'][i]) > 40:  # Filter out low-confidence noise
        line_num = d['top'][i]   # Use the vertical 'top' position as a key
        text = d['text'][i]
        
        # Group text that is within 10 pixels of each other vertically
        found = False
        for existing_top in lines.keys():
            if abs(line_num - existing_top) < 10: 
                lines[existing_top].append((d['left'][i], text))
                found = True
                break
        if not found:
            lines[line_num] = [(d['left'][i], text)]

# Sort words by their horizontal (left) position and print
for top in sorted(lines.keys()):
    sorted_line = sorted(lines[top], key=lambda x: x[0])
    print(" ".join([w[1] for w in sorted_line]))

Republic of the Philippines
Municipality of Infanta
Office of the Municipal Mayor
Business Permit
To whom it may concern,
Pursuant to the revenue code of this Municipality/City, after payment of taxes, fees and charges, etc., and compliance
with existing requirements, Permit is hereby granted to the herein Taxpayer.
BUSINESS NAME OWNER'SNAME BUSINESS ID NO. eNO. BUSINESS PERMIT NO.
— — — —
LMAS ENTERPRISES 2026-0405620000-0306
OG
| |
OFFICIAL RECEIPT
DATE ISSUED BUSINESS PAYMENT MODE OR DATE
rr — - =
ten. 2026
9783465 18,
2026-02-06 Sole Proprietorship Annual
9783467
Jan. 18, 2026
| TYPE OF
BUSINESS ADDRESS LINE OF BUSINESS
| APPLICATION.
aps ee
“OTHER WHOLESALE OF FOOD, AND
Renewal | BEVERAGES TOBACCO
PUROK LANGKA GUMIAN, INFANTA, QUEZON
TI. REGISTRATION NO. OF EMPLOYEES
UNTIL
Dec. 31, 2026 | 05005968
187-696-219-000
KIND OF FEE/TAX AMOUNT NOTE/S
Mayor's Permit Fee (asset Size)
700.00 NOTES:
Business Tax-Non Essential 1,756.70 Exhibit this
1. Permit in Your Establishment.
Health Certi

In [10]:
data

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text
4,5,1,1,1,1,1,1362,101,1175,3,95.0,
8,5,1,2,1,1,1,1387,101,1150,3,95.0,
12,5,1,3,1,1,1,1387,101,1150,3,95.0,
16,5,1,4,1,1,1,1387,101,1150,3,95.0,
20,5,1,5,1,1,1,1387,101,1150,3,95.0,
...,...,...,...,...,...,...,...,...,...,...,...,...
624,5,1,156,1,1,1,2348,4958,652,7,95.0,
628,5,1,157,1,1,1,3383,4962,469,1,95.0,
632,5,1,158,1,1,1,3383,4962,469,1,95.0,
636,5,1,159,1,1,1,635,4963,385,3,95.0,
